# Paper-1 Runner — ejecución secuencial de las 5 fases

Notebook para ejecutar en vivo las 5 fases del Paper-1 viendo el progreso en stdout.

**Fases en orden obligatorio:**
1. `phase1_train_hmm_seeds.py` — entrena HMM K∈{3..10} × 3 seeds (copia legacy seed=42 + entrena seeds 2021 y 7).
2. `phase2_k_sweep_seeds.py` — K-sweep downstream con 3 seeds. Output: `scripts/paper1/k_optimal.json`.
3. `phase3_plan_a_seeds.py` — Plan A completo 6 técnicas × 4 datasets × 4 horizontes × 3 seeds.
4. `phase4_cross_domain_seeds.py` — Cross-domain Traffic/Exchange × 4 fuentes HMM × 3 seeds.
5. `phase5_intrinsic_metrics.py` — métricas intrínsecas 6 técnicas × 4 datasets × 3 seeds.

**Todas las fases son resumibles** (skip si output existe). Puedes interrumpir y reanudar con la misma celda.

**Plan completo**: ver `memoria/secciones/plan_paper.md`.

---

## Estimación de coste CPU total

| Fase | Experimentos nuevos | Tiempo CPU aprox. |
|---|---|---|
| Phase 1 (HMM train) | 64 Baum-Welch nuevos (seed=42 se copia) | ~6-10 h |
| Phase 2 (K-sweep) | 192 Transformer trains | ~15-25 h |
| Phase 3 (Plan A) | 288 Transformer trains | ~60-80 h |
| Phase 4 (Cross-domain) | 108 Transformer trains | ~25-35 h |
| Phase 5 (Intrinsic) | ~30 min total | ~30 min |
| **Total** | **~652 experimentos** | **~110-150 h CPU** |

En paralelo / con GPU los tiempos bajan 3-5x.

## 0. Setup — verificar repo, caches existentes, entorno

In [1]:
import os, sys
from pathlib import Path

REPO = '/home/jaime/TFG/RITMO'
if os.getcwd() != REPO:
    os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

print('CWD:', os.getcwd())
print('Python:', sys.executable)

# Verificar que los 4 CSVs existen.
from scripts.paper1.config import DATASETS, CROSS_DOMAIN_TARGETS, SEEDS, K_VALUES
print(f'\nSeeds: {SEEDS}')
print(f'K values: {K_VALUES}')
print(f'\nDatasets IN-DOMAIN:')
for ds in DATASETS:
    p = Path(ds['csv'])
    print(f"  {ds['name']:12} {ds['csv']:50} exists={p.exists()}")
print(f'\nDatasets CROSS-DOMAIN:')
for t in CROSS_DOMAIN_TARGETS:
    p = Path(t['root']) / t['data_path']
    print(f"  {t['name']:12} {str(p):50} exists={p.exists()}")

CWD: /home/jaime/TFG/RITMO
Python: /home/jaime/anaconda3/envs/ritmo/bin/python

Seeds: [42, 2021, 7]
K values: [3, 4, 5, 6, 7, 8, 9, 10]

Datasets IN-DOMAIN:
  ETTh1        dataset/ETT-small/ETTh1.csv                        exists=True
  ETTh2        dataset/ETT-small/ETTh2.csv                        exists=True
  Weather      dataset/weather/weather.csv                        exists=True
  Electricity  dataset/electricity/electricity.csv                exists=True

Datasets CROSS-DOMAIN:
  Traffic      dataset/traffic/traffic.csv                        exists=True
  Exchange     dataset/exchange_rate/exchange_rate.csv            exists=True


In [2]:
# Verificar caches HMM legacy disponibles (seed=42 implícita).
from scripts.paper1.config import hmm_cache_path_legacy, K_VALUES, DATASETS

missing = []
for ds in DATASETS:
    for K in K_VALUES:
        p = Path(hmm_cache_path_legacy(ds['name'], K))
        if not p.exists():
            missing.append((ds['name'], K, str(p)))

if missing:
    print(f'FALTAN {len(missing)} caches legacy (se entrenarán desde cero con seed=42 en phase1):')
    for m in missing[:10]:
        print(f'  {m}')
else:
    print(f'OK: las 32 caches legacy (4 datasets × 8 Ks) existen.')

OK: las 32 caches legacy (4 datasets × 8 Ks) existen.


## Phase 1 — Entrenar HMM K∈{3..10} × 3 seeds

- **Input**: caches legacy en `cache/hmm_{ds}_K{k}.pth` (seed=42 implícita).
- **Output**: `cache/hmm_{ds}_K{k}_seed{s}.pth` para s ∈ {42, 2021, 7}.
- **Trabajo**: 64 Baum-Welch nuevos (seed=42 se copia desde legacy sin re-entrenar).
- **Resumible**: skip si el archivo existe.

Tiempo estimado CPU: **6-10 horas**. Puedes ejecutar en background:
```bash
nohup python -u scripts/paper1/phase1_train_hmm_seeds.py > logs/paper1_phase1.log 2>&1 &
tail -f logs/paper1_phase1.log
```

In [3]:
# Dry-run: ver qué haría sin entrenar.
!python -u scripts/paper1/phase1_train_hmm_seeds.py --dry-run 2>&1 | tail -40

[61/96] SKIP ./cache/hmm_weather_K7_seed42.pth
[62/96] SKIP ./cache/hmm_weather_K7_seed2021.pth
[63/96] SKIP ./cache/hmm_weather_K7_seed7.pth
[64/96] SKIP ./cache/hmm_weather_K8_seed42.pth
[65/96] SKIP ./cache/hmm_weather_K8_seed2021.pth
[66/96] SKIP ./cache/hmm_weather_K8_seed7.pth
[67/96] SKIP ./cache/hmm_weather_K9_seed42.pth
[68/96] SKIP ./cache/hmm_weather_K9_seed2021.pth
[69/96] SKIP ./cache/hmm_weather_K9_seed7.pth
[70/96] SKIP ./cache/hmm_weather_K10_seed42.pth
[71/96] SKIP ./cache/hmm_weather_K10_seed2021.pth
[72/96] SKIP ./cache/hmm_weather_K10_seed7.pth

[paper1-phase1] === Electricity ===
[73/96] SKIP ./cache/hmm_electricity_K3_seed42.pth
[74/96] SKIP ./cache/hmm_electricity_K3_seed2021.pth
[75/96] SKIP ./cache/hmm_electricity_K3_seed7.pth
[76/96] SKIP ./cache/hmm_electricity_K4_seed42.pth
[77/96] SKIP ./cache/hmm_electricity_K4_seed2021.pth
[78/96] SKIP ./cache/hmm_electricity_K4_seed7.pth
[79/96] SKIP ./cache/hmm_electricity_K5_seed42.pth
[80/96] SKIP ./cache/hmm_electric

In [4]:
# Ejecución real: entrena todo lo faltante. Live output en el notebook.
!python -u scripts/paper1/phase1_train_hmm_seeds.py

[paper1-phase1] datasets=['ETTh1', 'ETTh2', 'Weather', 'Electricity'] Ks=[3, 4, 5, 6, 7, 8, 9, 10] seeds=[42, 2021, 7] total_expected=96

[paper1-phase1] === ETTh1 ===
[1/96] SKIP ./cache/hmm_etth1_K3_seed42.pth
[2/96] SKIP ./cache/hmm_etth1_K3_seed2021.pth
[3/96] SKIP ./cache/hmm_etth1_K3_seed7.pth
[4/96] SKIP ./cache/hmm_etth1_K4_seed42.pth
[5/96] SKIP ./cache/hmm_etth1_K4_seed2021.pth
[6/96] SKIP ./cache/hmm_etth1_K4_seed7.pth
[7/96] SKIP ./cache/hmm_etth1_K5_seed42.pth
[8/96] SKIP ./cache/hmm_etth1_K5_seed2021.pth
[9/96] SKIP ./cache/hmm_etth1_K5_seed7.pth
[10/96] SKIP ./cache/hmm_etth1_K6_seed42.pth
[11/96] SKIP ./cache/hmm_etth1_K6_seed2021.pth
[12/96] SKIP ./cache/hmm_etth1_K6_seed7.pth
[13/96] SKIP ./cache/hmm_etth1_K7_seed42.pth
[14/96] SKIP ./cache/hmm_etth1_K7_seed2021.pth
[15/96] SKIP ./cache/hmm_etth1_K7_seed7.pth
[16/96] SKIP ./cache/hmm_etth1_K8_seed42.pth
[17/96] SKIP ./cache/hmm_etth1_K8_seed2021.pth
[18/96] SKIP ./cache/hmm_etth1_K8_seed7.pth
[19/96] SKIP ./cache/hmm_

In [5]:
# Verificar que las 96 caches (4 ds × 8 K × 3 seeds) están disponibles.
from scripts.paper1.config import hmm_cache_path, K_VALUES, SEEDS, DATASETS

missing = []
for ds in DATASETS:
    for K in K_VALUES:
        for s in SEEDS:
            p = Path(hmm_cache_path(ds['name'], K, s))
            if not p.exists():
                missing.append((ds['name'], K, s))

total = len(DATASETS) * len(K_VALUES) * len(SEEDS)
print(f'Caches disponibles: {total - len(missing)}/{total}')
if missing:
    print(f'\nFaltan {len(missing)}:')
    for m in missing[:10]:
        print(f'  {m}')

Caches disponibles: 96/96


### Phase 1.5 — Verificar convergencia HMM (CRÍTICO)

Los scripts de baum_welch corren con `max_iter=2000`, pero si alguna cache no convergió los resultados del paper quedan invalidados. Esta celda verifica las 96 caches y re-entrena con `max_iter=5000` las que no convergieron.

**Sale con error (exit 1) si hay alguna no convergida o faltante.**

In [6]:
# Verificación: lista caches no convergidas, cerca del max_iter, y missing.
!python -u scripts/paper1/verify_hmm_convergence.py

Inspeccionadas 96/96 caches
  OK convergidas:     96
  NO convergidas:     0
  Cerca del max_iter: 0
  MISSING (no entrenadas aún): 0

=== Resumen LL por (dataset, K) sobre seeds ===
  ETTh1        K= 3  n=3  LL min=-7713.03  max=-7713.03  range=0.00
  ETTh1        K= 4  n=3  LL min=-6748.81  max=-6748.81  range=0.00
  ETTh1        K= 5  n=3  LL min=-6123.48  max=-6123.48  range=0.00
  ETTh1        K= 6  n=3  LL min=-5585.05  max=-5585.05  range=0.00
  ETTh1        K= 7  n=3  LL min=-5257.62  max=-5257.61  range=0.01
  ETTh1        K= 8  n=3  LL min=-4993.59  max=-4993.59  range=0.00
  ETTh1        K= 9  n=3  LL min=-4843.35  max=-4843.25  range=0.10
  ETTh1        K=10  n=3  LL min=-4730.35  max=-4730.35  range=0.00
  ETTh2        K= 3  n=3  LL min=-6967.04  max=-6967.04  range=0.00
  ETTh2        K= 4  n=3  LL min=-5748.58  max=-5748.58  range=0.00
  ETTh2        K= 5  n=3  LL min=-6804.52  max=-4806.41  range=1998.11
  ETTh2        K= 6  n=3  LL min=-4115.64  max=-4115.64  range=0.0

In [7]:
# Si alguna no convergió: re-entrena con max_iter=5000. Ejecutar solo si la celda anterior reportó no-convergidas.
!python -u scripts/paper1/verify_hmm_convergence.py --retrain-nonconverged

Inspeccionadas 96/96 caches
  OK convergidas:     96
  NO convergidas:     0
  Cerca del max_iter: 0
  MISSING (no entrenadas aún): 0

=== Resumen LL por (dataset, K) sobre seeds ===
  ETTh1        K= 3  n=3  LL min=-7713.03  max=-7713.03  range=0.00
  ETTh1        K= 4  n=3  LL min=-6748.81  max=-6748.81  range=0.00
  ETTh1        K= 5  n=3  LL min=-6123.48  max=-6123.48  range=0.00
  ETTh1        K= 6  n=3  LL min=-5585.05  max=-5585.05  range=0.00
  ETTh1        K= 7  n=3  LL min=-5257.62  max=-5257.61  range=0.01
  ETTh1        K= 8  n=3  LL min=-4993.59  max=-4993.59  range=0.00
  ETTh1        K= 9  n=3  LL min=-4843.35  max=-4843.25  range=0.10
  ETTh1        K=10  n=3  LL min=-4730.35  max=-4730.35  range=0.00
  ETTh2        K= 3  n=3  LL min=-6967.04  max=-6967.04  range=0.00
  ETTh2        K= 4  n=3  LL min=-5748.58  max=-5748.58  range=0.00
  ETTh2        K= 5  n=3  LL min=-6804.52  max=-4806.41  range=1998.11
  ETTh2        K= 6  n=3  LL min=-4115.64  max=-4115.64  range=0.0

## Phase 2 — K-sweep downstream con 3 seeds

- **Input**: caches HMM de fase 1.
- **Output**: `scripts/paper1/k_optimal.json` con (variant, K) óptimo robusto por dataset.
- **Trabajo**: 4 datasets × 8 Ks × 2 variantes × 3 seeds = **192 experimentos**.
- **Resumible**: skip si `results/plan_a_..._ksweep_paper1_..._0/metrics.npy` existe.

Tiempo estimado CPU: **15-25 horas**. Background:
```bash
nohup python -u scripts/paper1/phase2_k_sweep_seeds.py > logs/paper1_phase2.log 2>&1 &
tail -f logs/paper1_phase2.log
```

In [8]:
# Opcional: correr solo un subset para test rápido (ejemplo: ETTh1 K=4 seed=42).
# !python -u scripts/paper1/phase2_k_sweep_seeds.py --only-dataset ETTh1 --only-ks 4 --only-seed 42

In [8]:
# Ejecución completa.
!python -u scripts/paper1/phase2_k_sweep_seeds.py

[paper1-phase2] datasets=['ETTh1', 'ETTh2', 'Weather', 'Electricity'] variants=['hmm_soft', 'hmm_soft_residual'] Ks=[3, 4, 5, 6, 7, 8, 9, 10] seeds=[42, 2021, 7] total=192

[paper1-phase2] === ETTh1 ===
[1/192] SKIP ETTh1 hmm_soft K=3 seed=42
[2/192] SKIP ETTh1 hmm_soft K=3 seed=2021
[3/192] SKIP ETTh1 hmm_soft K=3 seed=7
[4/192] SKIP ETTh1 hmm_soft K=4 seed=42
[5/192] SKIP ETTh1 hmm_soft K=4 seed=2021
[6/192] SKIP ETTh1 hmm_soft K=4 seed=7
[7/192] SKIP ETTh1 hmm_soft K=5 seed=42
[8/192] SKIP ETTh1 hmm_soft K=5 seed=2021
[9/192] SKIP ETTh1 hmm_soft K=5 seed=7
[10/192] SKIP ETTh1 hmm_soft K=6 seed=42
[11/192] SKIP ETTh1 hmm_soft K=6 seed=2021
[12/192] SKIP ETTh1 hmm_soft K=6 seed=7
[13/192] SKIP ETTh1 hmm_soft K=7 seed=42
[14/192] SKIP ETTh1 hmm_soft K=7 seed=2021
[15/192] SKIP ETTh1 hmm_soft K=7 seed=7
[16/192] SKIP ETTh1 hmm_soft K=8 seed=42
[17/192] SKIP ETTh1 hmm_soft K=8 seed=2021
[18/192] SKIP ETTh1 hmm_soft K=8 seed=7
[19/192] SKIP ETTh1 hmm_soft K=9 seed=42
[20/192] SKIP ETTh1 h

In [9]:
# Inspeccionar k_optimal.json.
import json
k_opt = json.loads(Path('scripts/paper1/k_optimal.json').read_text())
print('K óptimo robusto por dataset (avg MSE sobre 3 seeds):')
for name, best in k_opt.items():
    if best.get('variant'):
        mses = ', '.join(f"{x:.4f}" for x in best['mse_per_seed'])
        print(f"  {name:12} {best['variant']:20} K={best['K']}  avg={best['mse_avg']:.6f} "
              f"seeds=[{mses}] n={best['n_seeds']}")
    else:
        print(f'  {name:12} SIN DATOS')

K óptimo robusto por dataset (avg MSE sobre 3 seeds):
  ETTh1        hmm_soft             K=8  avg=0.059619 seeds=[0.0577, 0.0601, 0.0611] n=3
  ETTh2        hmm_soft             K=9  avg=0.145494 seeds=[0.1552, 0.1387, 0.1425] n=3
  Weather      hmm_soft_residual    K=4  avg=0.001243 seeds=[0.0013, 0.0012, 0.0012] n=3
  Electricity  hmm_soft_residual    K=3  avg=0.310024 seeds=[0.3065, 0.3171, 0.3064] n=3


## Phase 3 — Plan A completo con 3 seeds

- **Input**: `k_optimal.json` de fase 2.
- **Output**: `results/plan_a_..._final_paper1_{technique}_..._seed{s}_0/metrics.npy`.
- **Trabajo**: 4 datasets × 6 técnicas × 4 horizontes × 3 seeds = **288 experimentos**.
- **Resumible**.

Tiempo estimado CPU: **60-80 horas**. Background recomendado:
```bash
nohup python -u scripts/paper1/phase3_plan_a_seeds.py > logs/paper1_phase3.log 2>&1 &
```

In [11]:
# Opcional: correr solo un subset (ejemplo ETTh1 horizonte 96 seed 42).
# !python -u scripts/paper1/phase3_plan_a_seeds.py --only-dataset ETTh1 --only-horizons 96 --only-seed 42

In [ ]:
# Ejecución completa.
!python -u scripts/paper1/phase3_plan_a_seeds.py

[paper1-phase3] datasets=['ETTh1', 'ETTh2', 'Weather', 'Electricity'] seeds=[42, 2021, 7] horizons=[96, 192, 336, 720] total=288

[paper1-phase3] === ETTh1 ===
[1/288] SKIP ETTh1 discretization pl=96 seed=42
[2/288] SKIP ETTh1 discretization pl=96 seed=2021
[3/288] SKIP ETTh1 discretization pl=96 seed=7
[4/288] SKIP ETTh1 discretization pl=192 seed=42
[5/288] SKIP ETTh1 discretization pl=192 seed=2021
[6/288] SKIP ETTh1 discretization pl=192 seed=7
[7/288] SKIP ETTh1 discretization pl=336 seed=42
[8/288] SKIP ETTh1 discretization pl=336 seed=2021
[9/288] SKIP ETTh1 discretization pl=336 seed=7
[10/288] SKIP ETTh1 discretization pl=720 seed=42
[11/288] SKIP ETTh1 discretization pl=720 seed=2021
[12/288] SKIP ETTh1 discretization pl=720 seed=7
[13/288] SKIP ETTh1 text_based pl=96 seed=42
[14/288] SKIP ETTh1 text_based pl=96 seed=2021
[15/288] SKIP ETTh1 text_based pl=96 seed=7
[16/288] SKIP ETTh1 text_based pl=192 seed=42
[17/288] SKIP ETTh1 text_based pl=192 seed=2021
[18/288] SKIP ETTh

## Phase 4 — Cross-domain Traffic/Exchange con 3 seeds

- **Input**: `k_optimal.json` + caches HMM.
- **Output**: `results/plan_a_..._crossdom_paper1_..._0/metrics.npy`.
- **Trabajo**: 2 targets × (5 baselines + 4 fuentes HMM) × 2 horizontes × 3 seeds = **108 experimentos**.
- **Resumible**.

Tiempo estimado CPU: **25-35 horas**.

In [ ]:
# Ejecución completa.
!python -u scripts/paper1/phase4_cross_domain_seeds.py

## Phase 5 — Métricas intrínsecas de tokenización con 3 seeds

- **Input**: caches HMM + `k_optimal.json`.
- **Output**: `results/paper1_intrinsic/{dataset}_seed{s}.json` con las 11 métricas por técnica.
- **Trabajo**: 4 datasets × 3 seeds = 12 configuraciones × 6 técnicas.
- **Resumible**.

Tiempo estimado CPU: **~30 min total** (mucho más barato que las fases downstream).

In [ ]:
# Ejecución completa.
!python -u scripts/paper1/phase5_intrinsic_metrics.py

In [ ]:
# Quick look a los resultados intrínsecos (ejemplo ETTh1 seed 42).
import json
fp = Path('results/paper1_intrinsic/ETTh1_seed42.json')
if fp.exists():
    data = json.loads(fp.read_text())
    for tech, metrics in data.items():
        if tech.startswith('_'):
            continue
        keys = ['compression_ratio', 'mse_reconstruction', 'acf_retention']
        vals = ', '.join(f'{k}={metrics.get(k, "-"):.4f}' if isinstance(metrics.get(k), (int, float)) else f'{k}={metrics.get(k, "-")}' for k in keys)
        print(f'{tech:20} {vals}')
else:
    print(f'{fp} no existe todavía. Ejecuta phase5.')

## Resumen de estado

Celda final para ver el progreso global del Paper-1 en cualquier momento.

In [ ]:
# Dashboard compacto: cuántos outputs por fase existen vs esperados.
from scripts.paper1.config import (
    DATASETS, CROSS_DOMAIN_TARGETS, K_VALUES, SEEDS, HORIZONS,
    HMM_VARIANTS, BASELINE_TECHNIQUES, CROSS_DOMAIN_HORIZONS, hmm_cache_path,
)

def count_files(pattern):
    return len(list(Path('.').glob(pattern)))

# Phase 1: caches HMM
expected_p1 = len(DATASETS) * len(K_VALUES) * len(SEEDS)
got_p1 = sum(
    Path(hmm_cache_path(ds['name'], K, s)).exists()
    for ds in DATASETS for K in K_VALUES for s in SEEDS
)
print(f'Phase 1 (HMM caches):      {got_p1}/{expected_p1}')

# Phase 2: K-sweep downstream
expected_p2 = len(DATASETS) * len(K_VALUES) * len(HMM_VARIANTS) * len(SEEDS)
got_p2 = count_files('results/plan_a_*_ksweep_paper1_*_0/metrics.npy')
print(f'Phase 2 (K-sweep runs):    {got_p2}/{expected_p2}')

# Phase 3: Plan A
expected_p3 = len(DATASETS) * (len(BASELINE_TECHNIQUES) + 1) * len(HORIZONS) * len(SEEDS)
got_p3 = count_files('results/plan_a_*_final_paper1_*_0/metrics.npy')
print(f'Phase 3 (Plan A runs):     {got_p3}/{expected_p3}')

# Phase 4: Cross-domain
expected_p4 = (
    len(CROSS_DOMAIN_TARGETS) * len(BASELINE_TECHNIQUES) * len(CROSS_DOMAIN_HORIZONS) * len(SEEDS)
    + len(CROSS_DOMAIN_TARGETS) * len(DATASETS) * len(CROSS_DOMAIN_HORIZONS) * len(SEEDS)
)
got_p4 = count_files('results/plan_a_*_crossdom_paper1_*_0/metrics.npy')
print(f'Phase 4 (Cross-domain):    {got_p4}/{expected_p4}')

# Phase 5: Intrinsic metrics
expected_p5 = len(DATASETS) * len(SEEDS)
got_p5 = count_files('results/paper1_intrinsic/*_seed*.json')
print(f'Phase 5 (Intrinsic):       {got_p5}/{expected_p5}')

# k_optimal.json
k_opt_exists = Path('scripts/paper1/k_optimal.json').exists()
print(f'\nk_optimal.json:            {"OK" if k_opt_exists else "FALTA (corre phase 2)"}')